# 07 · Visual-token compression (stage S4)

Every method keeps a fraction `k` of the visual tokens before the decoder, through one call, `api.compress`, chosen by
`cfg["compressor"]`. This notebook checks each method on one GQA and one TextVQA dev item. Nothing here is a result:
it shows that the kept count is exact, that `k = 100%` changes nothing, and what the model answers.

| Method | Chooses tokens by | Code |
|---|---|---|
| `random`, `uniform`, `text_mean` | chance / even spacing / similarity to the mean question embedding | ours (controls) |
| `lowres` | nothing: a smaller image with about `k` times the tokens | config only (control) |
| `crisp_lite` | similarity to the question's nouns, plus a uniform fill | reimplemented from the CRISP paper |
| `rediprune` | relevance to the question and diversity | official |
| `divprune` | diversity only | official |
| `visionzip` | attention in the vision tower; the rest is merged | official |
| `cls_iqr` | CLS attention outliers (TRIM IQR rule on CLS attention, not TRIM); the count differs per image | rule ported from TRIM; Step-7 only |

Details and deviations: `docs/notes/compression_baselines.md`.

**How to run:** Run All, no edits, then paste the Step-4 output. `ALL_GPUS = True` (first code cell) needs GPUs 0-7 free and loads 4 model copies; Step-6 and Step-7 split their items over them, the single-item steps and the latency passes use one copy. Set it to False to stay on GPUs 0-3. Needs spaCy with `en_core_web_sm`. Step-6 (S5) and Step-7 (S5b Pareto sweep) are behind their own flags.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import full_gpu_overrides, select_for_notebook

CONFIG = "configs/mvp.yaml"
ALL_GPUS = True  # this notebook only: 4 model copies on GPUs 0-7; False: 0-3 with the config's n_copies
GPU_CFG = full_gpu_overrides(CONFIG) if ALL_GPUS else {}  # unhashed placement keys
GPU_IDS, N_COPIES = select_for_notebook(CONFIG, GPU_CFG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook(CONFIG, overrides=GPU_CFG)
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

ON_GPU_HOST = api.is_gpu_host()  # Inara or PACE
print("host", socket.gethostname(), "| on Inara or PACE:", ON_GPU_HOST)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd
import spacy

In [ ]:
# Settings
# cls_iqr has no keep fraction (its knob is m), so it is not in the k checks below; Step-7 runs it
METHODS = [m for m in api.METHODS if m not in ("none", "cls_iqr")]
KS = [0.25, 0.5]  # Step-4 one-item check only; the Pareto budgets are cfg["pareto"]["ks"] (Step-7)
R = 3  # loop passes
NOUN_MODEL = cfg["compression"]["crisp_lite"]["spacy_model"]
METHODS, cfg["compression"]

### Step-1:- Load the model and the noun tagger

`crisp_lite` needs spaCy's small English model; loading it first fails fast if it is missing.

In [ ]:
print("spaCy", spacy.__version__, "|", [t.text for t in spacy.load(NOUN_MODEL)("What color is the car?") if t.pos_ == "NOUN"])

In [ ]:
print("[STAGE] load model")
vlm = api.load_models(cfg, N_COPIES)[0]
api.set_loops(vlm, R)

### Step-2:- Two items and the uncompressed reference

The reference is the model with every visual token and no compression step.

In [ ]:
items = [api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0], api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]
[(it.benchmark, it.question, it.answers[:3]) for it in items]

In [ ]:
def run(it, method, k):
    """Compress one item with `method` at keep fraction `k`, then generate."""
    run_cfg = {**cfg, "compressor": method}
    inputs, info = api.compress(vlm, api.item_inputs(vlm, it, run_cfg), it, k, run_cfg)
    return api.generate_answers(vlm, [inputs], run_cfg)[0], info


print("[STAGE] reference")
reference = {}
for it in items:
    gen, info = run(it, "none", 1.0)
    reference[it.item_id] = gen
    print(f"{it.benchmark} | visual tokens {info['n_vis']} (grid {info['grid_hw']}) | raw {gen.raw_text!r} | "
          f"pred {api.read_answer(gen, cfg)!r} | gold {it.answers[:3]}")

### Step-3:- k = 100% changes nothing

At `k = 100%` a method keeps everything but still takes its packed path (vision tower run outside `generate`), so equal
token ids check the plumbing.

In [ ]:
print("[STAGE] k = 100%")
for method in METHODS:
    for it in items:
        gen, info = run(it, method, 1.0)
        same = torch.equal(gen.token_ids, reference[it.item_id].token_ids)
        assert same and info["n_vis"] == info["n_vis_full"], f"{method} {it.benchmark}: k=1 changed the output {gen.raw_text!r}"
    print(f"✓ {method}: same generated ids as no compression on both items")

### Step-4:- Each method at 25% and 50% (paste this output)

Per method, `k` and item: kept tokens of the full count, the raw output, the parsed answer and the references.
`lowres` cannot hit an exact count (the grid grows a row or column at a time).

In [ ]:
print("[STAGE] methods")
rows = []
for method in METHODS:
    for k in KS:
        for it in items:
            gen, info = run(it, method, k)
            pred = api.read_answer(gen, cfg)
            rows.append({"method": method, "k": k, "benchmark": it.benchmark, "n_vis": info["n_vis"],
                         "n_vis_full": info["n_vis_full"], "grid_hw": info["grid_hw"], "res_px": info["res_px"],
                         "pred": pred, "correct": api.score(it.benchmark, pred, it.answers),
                         "vit_s": info["vit_s"], "compression_s": info["compression_s"],
                         "compression_flops": info["compression_flops"], "fallback": info["fallback"]})
            note = f" | fallback {info['fallback']}" if info["fallback"] else ""
            print(f"{method:10s} k={k:.2f} {it.benchmark:7s} | kept {info['n_vis']}/{info['n_vis_full']}{note}")
            print(f"    raw {gen.raw_text!r} | pred {pred!r} | gold {it.answers[:3]}")
table = pd.DataFrame(rows)

In [ ]:
table[["method", "k", "benchmark", "n_vis", "n_vis_full", "res_px", "correct", "vit_s", "compression_s", "compression_flops"]]

In [ ]:
for r in table.itertuples():
    want = api.n_keep_for(r.n_vis_full, r.k)
    if r.method == "lowres":
        assert abs(r.n_vis - want) <= max(r.grid_hw), f"lowres {r.benchmark} k={r.k}: {r.n_vis} vs {want}, grid {r.grid_hw}"
    else:
        assert r.n_vis == want, f"{r.method} {r.benchmark} k={r.k}: kept {r.n_vis}, want {want}"
print(f"✓ kept counts: exact for every method at {KS} (lowres within one grid row)")

### Step-5:- Controller features, one item first (stage S4b)

For every cell (k × R) the controller reads, after the last pass: the last-token state `h_last`, how much it moved
(`delta_norm`), the attention on the visual tokens (`vis_attn_mass`), and from the first answer-token logits the
`entropy`, `top1_prob` and `kl_prev` (against the R-1 run). `kl_prev` and `delta_norm` are NaN at R = 1. The logit
features cost a coda call when read early; that cost is stored with them (`flops_readout`).

In [ ]:
FEATURE_METHOD = cfg["compressor"] if cfg["compressor"] != "none" else "text_mean"  # compressor frozen to divprune (S5); text_mean only if the config says none
feature_cfg = {**cfg, "compressor": FEATURE_METHOD}
FEATURE_DIR = ROOT / cfg["out_dir"] / "nb07" / "features" / "base" / api.config_hash(feature_cfg)
print(FEATURE_METHOD, "| k", cfg["k_grid"], "| R", cfg["R_grid"], "|", FEATURE_DIR)

In [ ]:
def feature_table(features):
    """One row per (k, R) cell, without the hidden state."""
    return pd.DataFrame([{"k": k, "R": p, **{n: v for n, v in f.items() if n != "h_last"}}
                         for (k, p), f in sorted(features.items())])


print("[STAGE] features, 1 item")
rows, features = api.run_item_cells(vlm, items[0], feature_cfg)
feature_table(features)

In [ ]:
cells = [(k, p) for k in cfg["k_grid"] for p in cfg["R_grid"]]
assert sorted(features) == sorted(cells) and len(rows) == len(cells), (sorted(features), len(rows))
for (k, p), f in features.items():
    assert torch.isfinite(f["h_last"]).all() and 0 <= f["vis_attn_mass"] <= 1 and 0 < f["top1_prob"] <= 1, (k, p)
    assert math.isnan(f["kl_prev"]) == (p == 1), (k, p, f["kl_prev"])
full_R3 = next(r for r in rows if r["k"] == 1.0 and r["R"] == 3)
api.set_loops(vlm, R)
assert full_R3["raw_output"] == reference[items[0].item_id].raw_text, "k=1, R=3 differs from the Step-2 reference"
print(f"✓ features: {len(cells)} cells, finite; the k=1 R=3 row answers like the reference")

#### Five items, saved

In [ ]:
N_FEATURE_ITEMS = 5
print("[STAGE] features, save")
feature_items = api.load_gqa(cfg, "dev", limit=3, root=ROOT) + api.load_textvqa(cfg, "dev", limit=2, root=ROOT)
for n, it in enumerate(feature_items, 1):
    rows, features = api.run_item_cells(vlm, it, feature_cfg)
    path = api.save_features(features, FEATURE_DIR, it.item_id)
    print(f"[{n}/{len(feature_items)}] {it.item_id} -> {path.name}")

In [ ]:
assert len(feature_items) == N_FEATURE_ITEMS
for it in feature_items:
    saved = api.load_features(FEATURE_DIR / f"{it.item_id}.pt")
    assert sorted(saved) == sorted(cells), (it.item_id, sorted(saved))
api.set_loops(vlm, R)
print(f"✓ features saved for {N_FEATURE_ITEMS} items x {len(cells)} cells")

### Step-6:- Which method? (stage S5)

Every method at k = 25% and 50%, plus the full model once, at R = 3 on dev. Per cell: accuracy, FLOPs (vision tower +
decoder + compression), latency and compression time, actual visual tokens.

Two flags in the next cell, both off: `RUN_SAMPLE` runs `sample_items` dev items per benchmark (look at it first),
`RUN_FULL_DEV` the whole dev split. Runs resume if stopped: finished rows are kept, whatever batch size or copy count
wrote them. Items are run in equal-length batches of at most `eval_batch_size` on `n_copies` copies (checked just below); the vision tower runs
once per item. Batched rows have no per-item latency, so timing comes from a second pass: `latency_items` items per benchmark at
batch 1 on one copy (no contention), in its own folder.

In [ ]:
RUN_SAMPLE = False  # set True after the Step-4 output is pasted and OK'd
RUN_FULL_DEV = False  # set True after the sample table is OK'd
COMPARE_N = None if RUN_FULL_DEV else cfg["sample_items"]  # None: the whole dev split
COMPARE_DIR = ROOT / cfg["out_dir"] / "nb07" / ("compare_dev" if RUN_FULL_DEV else "compare_sample")
RUN_COMPARE = RUN_SAMPLE or RUN_FULL_DEV
cfg["compare"], len(api.comparison_cells(cfg)), COMPARE_DIR

#### Batched against batch 1

Before any batched run: 16 dev items per benchmark in equal-length batches (`batch_grouping: equal_length`: equal visual
and other token counts, so no padding) and each alone, for three methods at k = 25% and the full model at k = 100%.
Same compressed inputs, so only the batching differs. The items are those of the dev split with the most equal-length
partners. The generated ids must be identical. If not, padding is not the (only) cause: keep `eval_batch_size: 1`, run
the diagnostic below and report. Run with `RUN_BATCH_CHECK = True` before changing `eval_batch_size`.

In [ ]:
RUN_BATCH_CHECK = False  # ran Nov 26: 13/128 batched outputs differ, so runs stay at batch 1
N_BATCH_CHECK = 16  # items per benchmark
CHECK_BATCH = 8  # batch size under test, whatever eval_batch_size is
CHECK_CELLS = [("rediprune", 0.25), ("visionzip", 0.25), ("text_mean", 0.25), ("none", 1.0)]
if RUN_BATCH_CHECK:
    print(f"[STAGE] equal-length batches against batch 1, R={cfg['compare']['R']}")
    check_items = []
    for load in (api.load_gqa, api.load_textvqa):
        check_items += api.batchable_items(vlm, load(cfg, "dev", root=ROOT), cfg, N_BATCH_CHECK)
    batch_check = pd.DataFrame(api.check_batched(vlm, check_items, cfg, CHECK_CELLS, cfg["compare"]["R"], CHECK_BATCH))
batch_check if RUN_BATCH_CHECK else "skipped: RUN_BATCH_CHECK is False"

In [ ]:
if RUN_BATCH_CHECK:
    assert len(batch_check) == len(check_items) * len(CHECK_CELLS)
    print(batch_check.drop_duplicates(["benchmark", "item_id"]).batch_n.value_counts().sort_index().to_string())
    batched_rows = batch_check[batch_check.batch_n > 1]
    assert len(batched_rows) >= len(batch_check) // 2, "too few rows were really batched to prove anything"
    mismatch = batched_rows[~batched_rows.ids_equal]
    assert mismatch.empty, f"batching changed {len(mismatch)} of {len(batched_rows)} outputs:\n{mismatch.to_string()}"
    print(f"✓ equal-length batches = batch 1: same generated ids on {len(batched_rows)} batched (item, method) rows")
else:
    print("skipped: RUN_BATCH_CHECK is False")

#### Why a batched output differs (diagnostic, off by default)

8 dev items (4 per benchmark), two methods at k = 25%. Each item runs alone (batch 1), as 8 copies of itself (batched,
no padding) and inside one mixed batch (batched, left-padded). `first_margin` = top-2 logit gap of the first answer
token at batch 1; `*_first_diff` = max |logit change| there; `*_flip_*` = the first generated token that differs.

How to read it: a flip always has `flip_margin <= 2 * flip_diff`, so judge the sizes. Small diffs and flips only at near
ties: BF16 numerics. Large diffs in `mixed` but not in `copies`: padding or row alignment. Any flip in `copies`:
batching alone changes outputs, and equal-length batching cannot fix it.

In [ ]:
RUN_BATCH_DIAG = True  # ~5 min
if RUN_BATCH_DIAG:
    print("[STAGE] batch diagnostic")
    diag_items = api.load_gqa(cfg, "dev", limit=4, root=ROOT) + api.load_textvqa(cfg, "dev", limit=4, root=ROOT)
    diag_cells = [("rediprune", 0.25), ("visionzip", 0.25)]
    batch_diag = pd.DataFrame(api.diagnose_batch(vlm, diag_items, cfg, diag_cells, cfg["compare"]["R"]))
    (ROOT / cfg["out_dir"] / "nb07").mkdir(parents=True, exist_ok=True)
    batch_diag.to_csv(ROOT / cfg["out_dir"] / "nb07" / "batch_diag.csv", index=False)
    assert len(batch_diag) == len(diag_items) * len(diag_cells)
    print(batch_diag[["first_margin", "copies_first_diff", "mixed_first_diff"]].describe().to_string())
    for name in ("copies", "mixed"):
        flips = batch_diag[batch_diag[f"{name}_flip_step"].notna()]
        print(f"{name}: {len(flips)} of {len(batch_diag)} rows differ from batch 1")
        if len(flips):
            print(flips[["method", "item_id", "n_pad", f"{name}_flip_step", f"{name}_flip_margin",
                         f"{name}_flip_diff"]].to_string(index=False))
    print(f"✓ batch diagnostic on {len(batch_diag)} (item, method) rows")
batch_diag if RUN_BATCH_DIAG else "skipped: RUN_BATCH_DIAG is False"

In [ ]:
if RUN_COMPARE:
    print(f"[STAGE] comparison, batch {cfg['eval_batch_size']}, {N_COPIES} copies")
    vlms = api.load_models(cfg, N_COPIES)
    compare_items = {"gqa": api.load_gqa(cfg, "dev", limit=COMPARE_N, root=ROOT),
                     "textvqa": api.load_textvqa(cfg, "dev", limit=COMPARE_N, root=ROOT)}
    compare_rows = api.run_comparison(vlms, compare_items, cfg, COMPARE_DIR)
len(compare_rows) if RUN_COMPARE else "skipped: both flags are False"

#### Latency pass: batch 1, one copy

The same cells on `latency_items` dev items per benchmark, one item at a time on the first copy only. Only its timing
columns are used (medians per cell): `latency_s` = vision + compression + generate, `t_prefill_ms` (time to first token;
for `none` and `lowres` it includes the vision tower), `t_decode_ms`.

In [ ]:
if RUN_COMPARE:
    LATENCY_DIR = COMPARE_DIR.parent / f"{COMPARE_DIR.name}_latency"
    N_LATENCY = cfg["latency_items"]
    print(f"[STAGE] latency pass, {N_LATENCY} items per benchmark, batch 1, 1 copy | {LATENCY_DIR}")
    latency_items = {"gqa": api.load_gqa(cfg, "dev", limit=N_LATENCY, root=ROOT),
                     "textvqa": api.load_textvqa(cfg, "dev", limit=N_LATENCY, root=ROOT)}
    latency_rows = api.run_comparison(vlms[:1], latency_items, cfg, LATENCY_DIR, batch_size=1)
    results = pd.DataFrame(api.comparison_table(compare_rows, latency_rows))
results.sort_values(["benchmark", "k", "accuracy"], ascending=[True, True, False]) if RUN_COMPARE else "skipped: both flags are False"

In [ ]:
if RUN_COMPARE:
    n_items = {b: len(v) for b, v in compare_items.items()}
    assert len(results) == 3 * len(api.comparison_cells(cfg)), len(results)
    assert all(r.n == (sum(n_items.values()) if r.benchmark == "all" else n_items[r.benchmark]) for r in results.itertuples())
    assert all(r.latency_n == (2 * N_LATENCY if r.benchmark == "all" else N_LATENCY) for r in results.itertuples()), "latency pass incomplete"
    assert (results.latency_s > 0).all() and results.t_prefill_ms.notna().all()
    full = results[results.method == "none"].set_index("benchmark").gflops_total
    pruned = results[(results.k < 1) & (results.method != "none")]
    assert all(r.gflops_total < full[r.benchmark] for r in pruned.itertuples()), "a pruned cell costs more than the full model"
    print(f"✓ comparison: every cell has every item and its latency; every pruned cell is cheaper than the full model")
else:
    print("skipped: both flags are False")

#### Winner

Best accuracy over k = 25% and 50%; methods whose paired 95% bootstrap interval against it contains 0 count as tied,
and the cheapest of those wins. A suggestion (the compressor was frozen to divprune from this run, Nov 25).
**Decide on the full-dev run only:** on the sample the intervals are wide, nearly everything ties, and the rule then
just picks the cheapest method (possibly a control).

In [ ]:
if RUN_COMPARE:
    winner, evidence = api.pick_winner(compare_rows, cfg)
    print("suggested compressor:", winner, "" if RUN_FULL_DEV else "(sample only: do not freeze it)")
pd.DataFrame(evidence) if RUN_COMPARE else "skipped: both flags are False"

In [ ]:
if RUN_COMPARE:
    api.save_table(results, "S5_compression" if RUN_FULL_DEV else "S5_compression_sample")
    print(f"## <date> · S5 · compression comparison (git {api.git_hash()}, dev n={n_items}, R={cfg['compare']['R']})")
    print(f"- Decision: compressor = {winner} because <reason>.")
    print(f"- Latency: median, batch 1, one copy, {N_LATENCY} items per benchmark.")
    print("| method | k | acc all | GFLOPs | latency s | prefill ms | decode ms | compression s | kept tokens |")
    print("|---|---|---|---|---|---|---|---|---|")
    for r in results[results.benchmark == "all"].sort_values(["k", "accuracy"], ascending=[True, False]).itertuples():
        print(f"| {r.method} | {r.k} | {r.accuracy:.1f} | {r.gflops_total:.1f} | {r.latency_s:.2f} | {r.t_prefill_ms:.0f} | {r.t_decode_ms:.0f} | {r.compression_s:.3f} | {r.n_vis:.0f} |")
else:
    print("skipped: both flags are False")

### Step-7:- Pareto sweep (stage S5b)

How does accuracy fall as the decoder gets fewer tokens? Every method at k = 10, 25, 50, 75% and the full model (100%),
plus `cls_iqr`, which decides its own count per image: it keeps the units whose CLS attention is above `Q3 + m * IQR`
(TRIM's rule on CLS attention, not TRIM; no extra token for the rest, our decoder never saw one). `m` is its knob, one
point per m. Dev, R = 3, batch 1, `N_COPIES` copies (4 with `ALL_GPUS`); settings: `pareto` in `configs/mvp.yaml`.

It runs at `res_px` of `configs/mvp.yaml` (768 since Nov 26): 38 cells, 9,728 answers on 256 items, and 1,216 for the
latency pass. A row already on disk with the cell's config hash is not run again, whether it is this sweep's (resume) or
a Step-6 file (k = 25 and 50% per method, the full model). The 512 px S5 files have other hashes, so they are not used.
The counts and a time estimate are printed before anything runs. After the sweep, the S5 rule is applied to its k = 25
and 50% rows to re-check the winner.

Set `RUN_PARETO = True` and Run All: first two items with `cls_iqr`, then one item per benchmark in every cell, then the
full run. Step-6's flags can stay False.

In [ ]:
RUN_PARETO = True  # set True to run Step-7
N_PARETO_SAMPLE = 1  # items per benchmark of the sample run
N_PARETO = cfg["pareto"]["n_items"]  # dev items per benchmark
PARETO_DIR = ROOT / cfg["out_dir"] / "nb07" / "pareto"
PARETO_LATENCY_DIR = ROOT / cfg["out_dir"] / "nb07" / "pareto_latency"
S5_DONE_DIR = ROOT / cfg["out_dir"] / cfg["pareto"]["reuse_dir"]
S5_DONE_LATENCY_DIR = ROOT / cfg["out_dir"] / cfg["pareto"]["reuse_latency_dir"]
CELLS = api.pareto_cells(cfg)
print(f"{len(CELLS)} cells | res_px {cfg['res_px']}")
print("S5 folder:", S5_DONE_DIR, "| exists:", S5_DONE_DIR.is_dir(), "| latency exists:", S5_DONE_LATENCY_DIR.is_dir())
pd.DataFrame([{"cell": c["cell"], "method": c["method"], "k": c["keys"]["k"], "m": c["m"]} for c in CELLS])

#### `cls_iqr` on one item per benchmark

Kept units per m on the two Step-2 items. A larger m raises the bar, so fewer are kept (at least 1).

In [ ]:
if RUN_PARETO:
    print("[STAGE] cls_iqr, 2 items")
    iqr_rows = []
    for it in items:
        inputs = api.item_inputs(vlm, it, cfg)
        vision = api.encode_vision(vlm, inputs)  # one vision run for every m
        for cell in [c for c in CELLS if c["method"] == "cls_iqr"]:
            _, info = api.compress(vlm, inputs, it, 1.0, {**cfg, **cell["keys"]}, vision=vision)
            iqr_rows.append({"benchmark": it.benchmark, "m": cell["m"], "n_vis": info["n_vis"],
                             "n_vis_full": info["n_vis_full"], "k": info["k"], "fallback": info["fallback"]})
    iqr = pd.DataFrame(iqr_rows)
else:
    print("skipped: RUN_PARETO is False")
iqr if RUN_PARETO else None

In [ ]:
if RUN_PARETO:
    for bench, group in iqr.groupby("benchmark"):
        counts = group.sort_values("m").n_vis.tolist()
        assert counts == sorted(counts, reverse=True) and counts[-1] >= 1, (bench, counts)
        assert counts[0] < group.n_vis_full.iloc[0] and (group.k == group.n_vis / group.n_vis_full).all(), (bench, counts)
    print("✓ cls_iqr: fewer tokens as m grows, k = kept / full")
else:
    print("skipped: RUN_PARETO is False")

#### What is left to run

Counted from the files: answers of the accuracy run and of the latency pass (its first `latency_items` items per
benchmark) that are not on disk yet; 0 when all are done. The time is rough: seconds per answer x answers, spread over
the copies (one copy for latency), without loading the model or preparing the images.

In [ ]:
if RUN_PARETO:
    pareto_items = {"gqa": api.load_gqa(cfg, "dev", limit=N_PARETO, root=ROOT),
                    "textvqa": api.load_textvqa(cfg, "dev", limit=N_PARETO, root=ROOT)}
    N_PARETO_LATENCY = cfg["latency_items"]
    latency_items = {b: v[:N_PARETO_LATENCY] for b, v in pareto_items.items()}
    seconds, seconds_source = api.pareto_seconds_per_answer(cfg, ROOT / cfg["out_dir"] / cfg["pareto"]["timing_dir"])
    left_rows = []
    for name, run_items, out_dir, done_dir, copies in (
            ("accuracy run", pareto_items, PARETO_DIR, S5_DONE_DIR, N_COPIES),
            ("latency pass", latency_items, PARETO_LATENCY_DIR, S5_DONE_LATENCY_DIR, 1)):
        status = pd.DataFrame(api.pareto_status(run_items, cfg, out_dir, done_dir))
        left_rows.append({"run": name, **status[["reused", "own", "missing"]].sum().to_dict(), "copies": copies,
                          "minutes": round(status.missing.sum() * seconds / copies / 60, 1)})
    left = pd.DataFrame(left_rows)
    print(f"{seconds:.2f} s per answer ({seconds_source})")
    print(f"answers missing: {left.missing.sum()} | about {left.minutes.sum():.0f} min")
else:
    print("skipped: RUN_PARETO is False")
left if RUN_PARETO else None

#### Sample run: one item per benchmark

Every cell on one item per benchmark. New rows are kept: the full run does not repeat them.

In [ ]:
if RUN_PARETO:
    vlms = api.load_models(cfg, N_COPIES)
    print(f"[STAGE] Pareto sample, {N_PARETO_SAMPLE} item per benchmark")
    sample_items = {b: v[:N_PARETO_SAMPLE] for b, v in pareto_items.items()}
    pareto_sample = pd.DataFrame(api.run_pareto(vlms, sample_items, cfg, PARETO_DIR, S5_DONE_DIR))
else:
    print("skipped: RUN_PARETO is False")
pareto_sample[["cell", "benchmark", "source", "n_vis", "n_vis_full", "k", "pred", "correct", "fallback"]] if RUN_PARETO else None

In [ ]:
def check_pareto(rows, run_items):
    """Every cell has every item; rows with an S5 file of their hash were read from it; kept counts per method."""
    assert len(rows) == len(CELLS) * sum(len(v) for v in run_items.values()) and not rows.duplicated(["cell", "benchmark", "item_id"]).any(), len(rows)
    status = pd.DataFrame(api.pareto_status(run_items, cfg, PARETO_DIR, S5_DONE_DIR))
    assert (status.missing == 0).all(), status[status.missing > 0]
    reused = rows[rows.source == "reused"].groupby(["cell", "benchmark"]).size()
    assert all(reused.get((s.cell, s.benchmark), 0) == s.reused for s in status.itertuples()), "an S5 row was run again"
    for r in rows.itertuples():
        if r.selector == "cls_iqr":
            assert 1 <= r.n_vis <= r.n_vis_full and abs(r.k - r.n_vis / r.n_vis_full) < 1e-9, (r.cell, r.item_id, r.n_vis)
        elif r.selector == "lowres":  # the grid grows a row or column at a time
            want = api.n_keep_for(r.n_vis_full, r.k)
            assert abs(r.n_vis - want) <= max(r.grid_hw), (r.cell, r.item_id, r.n_vis, want, r.grid_hw)
        else:
            assert r.n_vis == api.n_keep_for(r.n_vis_full, r.k), (r.cell, r.item_id, r.n_vis)


if RUN_PARETO:
    check_pareto(pareto_sample, sample_items)
    print(f"✓ sample: {len(CELLS)} cells, {(pareto_sample.source == 'reused').sum()} rows read from S5, kept counts as expected")
else:
    print("skipped: RUN_PARETO is False")

#### Full run: dev items of both benchmarks

In [ ]:
if RUN_PARETO:
    print(f"[STAGE] Pareto sweep, {N_PARETO} items per benchmark, batch 1, {N_COPIES} copies")
    pareto_rows = api.run_pareto(vlms, pareto_items, cfg, PARETO_DIR, S5_DONE_DIR)
    pareto_frame = pd.DataFrame(pareto_rows)
else:
    print("skipped: RUN_PARETO is False")
pareto_frame.groupby(["cell", "source"]).size().unstack(fill_value=0) if RUN_PARETO else None

In [ ]:
if RUN_PARETO:
    check_pareto(pareto_frame, pareto_items)
    print("✓ sweep: every cell has every item with the expected kept count")
else:
    print("skipped: RUN_PARETO is False")

#### The S5 winner again, at this size

Accuracy of every method at k = 25 and 50%, per benchmark and over both, and the S5 rule (`api.pick_winner`) on these
rows. It only reports: `compressor` in `configs/mvp.yaml` is changed by decision, not by this cell.

In [ ]:
if RUN_PARETO:
    winner_rows = [r for r in pareto_rows
                   if r["selector"] in cfg["compare"]["methods"] and r["k"] in cfg["compare"]["ks"]]
    winner_accuracy = pd.DataFrame(api.comparison_table(winner_rows)).pivot(
        index="method", columns=["k", "benchmark"], values="accuracy")
    winner_new, winner_evidence = api.pick_winner(winner_rows, cfg)
    print(f"accuracy (%), res_px {cfg['res_px']}, R={cfg['pareto']['R']}, n={N_PARETO} per benchmark")
    print(winner_accuracy.round(1).to_string())
    print(pd.DataFrame(winner_evidence).round(1).to_string(index=False))
    if winner_new == cfg["compressor"]:
        print(f"S5 rule at {cfg['res_px']} px: {winner_new} -> confirms {cfg['compressor']}")
    else:
        print(f"S5 rule at {cfg['res_px']} px: {winner_new} -> a DIFFERENT winner than {cfg['compressor']} "
              "(compressor not changed; report it)")
else:
    print("skipped: RUN_PARETO is False")

#### Latency pass and table

Cells are timed on `latency_items` dev items per benchmark, batch 1, one copy. A row found in the S5 latency folder with
the same hash is taken from there: an **earlier session** (`latency_from` = reused), so compare such latencies with care.
`n_vis` is the mean kept count (for `cls_iqr` it is the result of m), `k` the mean kept / full.

In [ ]:
if RUN_PARETO:
    print(f"[STAGE] latency pass, {N_PARETO_LATENCY} items per benchmark, 1 copy | {PARETO_LATENCY_DIR}")
    pareto_latency = api.run_pareto(vlms[:1], latency_items, cfg, PARETO_LATENCY_DIR, S5_DONE_LATENCY_DIR)
    pareto_results = pd.DataFrame(api.pareto_table(pareto_rows, pareto_latency, cfg))
else:
    print("skipped: RUN_PARETO is False")
PARETO_COLUMNS = ["method", "m", "k", "benchmark", "n", "accuracy", "gflops_total", "n_vis", "kv_cache_bytes", "latency_s",
                  "latency_from", "fallback_rate"]
pareto_results[PARETO_COLUMNS].sort_values(["benchmark", "method", "k"]) if RUN_PARETO else None

In [ ]:
if RUN_PARETO:
    assert len(pareto_results) == 3 * len(CELLS), len(pareto_results)
    assert (pareto_results[pareto_results.benchmark != "all"].latency_n == N_PARETO_LATENCY).all(), "latency pass incomplete"
    assert (pareto_results.latency_s > 0).all()
    timed = pd.DataFrame(api.pareto_status(latency_items, cfg, PARETO_LATENCY_DIR, S5_DONE_LATENCY_DIR))
    timed = timed.groupby("cell")[["n_items", "reused"]].sum()
    for r in pareto_results.itertuples():  # "reused" only where an S5 latency file of this hash exists
        n, old = timed.loc[r.cell]
        want = "+".join(name for name, count in (("reused", old), ("run", n - old)) if count)
        assert r.latency_from == want, (r.cell, r.latency_from, want)
    full_kv = pareto_results[pareto_results.method == "none"].set_index("benchmark").kv_cache_bytes
    pruned = pareto_results[pareto_results.method != "none"]
    assert all(r.kv_cache_bytes < full_kv[r.benchmark] for r in pruned.itertuples()), "a pruned cell is not smaller"
    print("✓ table: every cell has accuracy, cost and latency; every pruned cell has a smaller KV cache than full")
else:
    print("skipped: RUN_PARETO is False")

#### What `cls_iqr` keeps, and the overall Pareto front (paste these)

Mean kept units per m and benchmark; then the cells no other cell beats in both accuracy and GFLOPs (both benchmarks).

In [ ]:
pareto_results[pareto_results.method == "cls_iqr"].pivot(index="m", columns="benchmark", values="n_vis") if RUN_PARETO else None

In [ ]:
if RUN_PARETO:
    overall = pareto_results[pareto_results.benchmark == "all"].sort_values("gflops_total")
    front = overall[api.pareto_front(overall.gflops_total.tolist(), overall.accuracy.tolist())]
    api.save_table(pareto_results, "S5b_pareto")
    print(f"## <date> · S5b · Pareto sweep (git {api.git_hash()}, res_px {cfg['res_px']}, dev n={N_PARETO} per benchmark, R={cfg['pareto']['R']}, batch 1)")
    print("- cls_iqr = TRIM IQR rule on CLS attention (not TRIM), no extra token.")
    print(f"- S5 winner re-check (k = 25/50%): {winner_new} (current compressor: {cfg['compressor']}).")
    if (pareto_results.latency_from != "run").any():
        print("- Cells with latency_from = reused are timed in an earlier session (S5 files).")
else:
    print("skipped: RUN_PARETO is False")
front[PARETO_COLUMNS] if RUN_PARETO else None

#### Figure F06

Accuracy against GFLOPs, kept visual tokens and KV cache size, one line per method; the star is the full model, ringed
points are on that panel's Pareto front.

In [ ]:
if RUN_PARETO:
    api.set_paper_style()
    fig = api.plot_pareto(pareto_results)
else:
    print("skipped: RUN_PARETO is False")

In [ ]:
if RUN_PARETO:
    pdf_path, png_path = api.save_figure(fig, "F06_pareto", final=False)
    assert pdf_path.is_file() and png_path.is_file()
    print("✓ F06 draft saved")
else:
    print("skipped: F06 not saved")